____
### 0. Preamble
- This serves as a continuation to my other notebook files where models are trained in a plain environment
- Within this notebook file, I will be experimenting with a world model version of the 
- I will also be documenting my learnings from the paper : https://worldmodels.github.io/

____
### 1. Feedforward network vs Recurrent Neural Networks (RNN)

#### 1.2 Feedforward networks
- All neural networks implemented thus far are all **feedforward network** (`nn.Sequential` of `nn.Linear` layers). 
- **Feedforward networks** have no memory: given an input, they compute an output, and that's it(Nothing about the computation depends on what inputs came before.)
    - Feeding in the same `obs_t` in twice in a row, obtains the exact same output twice — there's no notion of "time" or "history" built into the architecture itself. 
    - This is fine for the PPO/TD3/SAC agents because the environment is already Markovian: 
    - `obs = [inventory, days_left, last_demand]` fully captures everything the policy needs to act optimally at that instant, so there's nothing useful left in the *history* of observations that isn't already summarized in the current one.

#### 1.3 Recurrent Neural Network (RNN)
- **Recurrent Neural Network (RNN)** carries a hidden state $h_t$ forward from one step to the next
    - The hidden state is updated as a function of both the new input and the previous hidden state:
            $$h_t = f(x_t, h_{t-1})$$
    - This means the network's output at time $t$ can depend on everything it has seen since the start of the sequence, not just the current input
    - Feed it the same $x_t$ twice in a row at two different points in a sequence, and you can get two different outputs, because $h_{t-1}$ was different each time.

| | Feedforward (e.g. `PPOActorCritic`) | Recurrent (e.g. `MDNRNN`) |
|---|---|---|
| Input → Output | `y = f(x)` | `y_t = f(x_t, h_{t-1})` |
| Memory | None — stateless | Hidden state $h_t$ carried across steps |
| Good for | Tasks where the current input already contains everything needed (Markovian state) | Tasks where what happened *before* matters — sequences, time series, language |
| In this notebook | The policy/value networks for PPO, TD3, SAC | The MDN-RNN world model (M), which needs to remember *trends* across many steps to predict the future |
    


```python
# Feedforward layer: stateless, same weights applied independently each call
linear = nn.Linear(in_features, out_features)
y = linear(x)                     # y depends only on x

# Recurrent layer: stateful, hidden state threads through calls
lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
out, (h, c) = lstm(x, (h_prev, c_prev))   # out depends on x AND h_prev
```

____
### 2. Context for World model

#### 2.1 Components of a World model
- There are 3 components to creating a world model 
    1. Vision Model (V)
        - Meant to encode overly complex environment state features into a single z vector
        - Can be used in reverse to encode z vectors ($z$) back into the complex environment 
        - $z_t$ refers to the z vector at timestep $t$
    2. Predictive Model (M)
        - Takes in input z vectors from V and generates future z vectors that V is expected to produce
        - Essentially predicting the next state of the environment
    3. Controller (C)
        - Reads the z vector and any hidden state vectors to produce an output action 


#### 2.2 Dream Environments
- Dream environments are created using M and decoded using V where necessary 
- How a Dream environment is constructed
    1. Produce a probability distribution of $z_{t+1}$ given a specefic $z_t$
    2. Sample a $z_{t+1}$ and use sample as a real observation
    3. Producing a distribution of $z_{t+1}$ creates a "dream environment"
- Therefore a Dream environment is created using the predictive model (M) which produces a state (represented by $z_t$)
    - predictive model (M) also needs to generate a binary indicator to indicate if the episode is done ($d_t$)
    - ie, M($z_t$, $d_t$) -> $z_{t+1}$, $d_{t+1}$
- Usually no V model is required to decode/encode the z vectors as the controller (C) model is able to read the state (only needs decoding for humans to read)

#### 2.3 Training inside Dreams
- Predictive Model (M) used to simulate RNN is RNN(Recurrent-Neural-Network) based
    - RNNs read a hidden state vector and essentially retains 'memory', generating the next z vector based on the previous states and hidden states (refer to the top)
- Thus RNNs help M to predict the next state ($z_{t+1}$, $d_{t+1}$) properly, mimicing a complete game environment
    - If environment consists of pixels, M will have to learn from only raw image data collected from random episodes
- The environment can be made more challenging for training by adjusting the temperature parameter $T$
    -  This is done by increasing $T$ during the sampling process of $z_{t+1}$

#### 2.4 Exploting Virtual Environment
1. M is only an appropriate probabilistic model of the environment (a probabilistic model that mimics the actual environment)
    - Occassionally generate states that don't follow the laws governing the actual environment
2. Controller is given full access to hidden states of M
    - Since the agent is trained using M, it is given access to the hidden states on top of the observable z vectors
    - Therefore the agent is granted access to all internal states and memory of the game engine instead of ONLY game observations
- Therefore agents can efficiently explore ways to directly manipulate hidden states of teh game engine when training to maximize expected cumulative results
- If model finds an adversarial policy that can fool the dynamics model  
    - The model looks good under virtual environment but performs poorly in an actual environment
    - This is because the model visits states where the model is wrong as they are away from training distribution (game states or states that are not within the regular training distribution)
3. Solution for Exploitation
    - use Mixture density RNN (MDNRNN) which approximates the environment to a stochastic one, even if it is deterministic
        - This allows controller (C) to train inside a stochastic environment
    - Adjust temperature parameter $T$ to control randomness
        - The higher the $T$ the more uncertain the environment is, preventing C from taking advantage of imperfections of world model
        - Higher $T$ also trains the model to handle noise better when thrown into the real environment

____
### 3. Imports

In [ ]:
import gymnasium as gym
import numpy as np
import torch as torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal
import copy

____
### 4. Import Original Environment